# EfficientNet-B0 졸음 분류 학습

이 노트북은 `y_model` 안의 `split_dataset`을 사용해 EfficientNet-B0 모델을 학습합니다.

In [ ]:
from pathlib import Path
import copy
import os
import random
import sys

import torch
from torch import nn
from torch.utils.data import DataLoader, WeightedRandomSampler
from torchvision import datasets, transforms
from torchvision.models import EfficientNet_B0_Weights, efficientnet_b0

# 재현 가능한 결과를 위해 시드를 고정합니다.
RANDOM_SEED = 42
random.seed(RANDOM_SEED)
torch.manual_seed(RANDOM_SEED)
if torch.cuda.is_available():
    torch.cuda.manual_seed_all(RANDOM_SEED)

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print('사용 장치 =', device)
print('torch 버전 =', torch.__version__)

In [ ]:
# 실행 위치가 달라도 split_dataset을 찾을 수 있게 후보 경로를 순서대로 확인합니다.
BASE_DIR = Path.cwd()
candidate_split_dirs = [
    BASE_DIR / 'split_dataset',
    BASE_DIR / 'captured_images_total' / 'split_dataset',
    BASE_DIR / 'y_model' / 'split_dataset',
    BASE_DIR / 'y_model' / 'captured_images_total' / 'split_dataset',
]

SPLIT_ROOT_DIR = next((path for path in candidate_split_dirs if path.exists()), candidate_split_dirs[0])
Y_MODEL_DIR = SPLIT_ROOT_DIR.parent if SPLIT_ROOT_DIR.name == 'split_dataset' else BASE_DIR
PROJECT_ROOT_DIR = Y_MODEL_DIR.parent if Y_MODEL_DIR.name == 'y_model' else BASE_DIR

TRAIN_DIR = SPLIT_ROOT_DIR / 'train'
VAL_DIR = SPLIT_ROOT_DIR / 'validation'
TEST_DIR = SPLIT_ROOT_DIR / 'test'

print('split root =', SPLIT_ROOT_DIR)
print('train 경로 =', TRAIN_DIR)
print('validation 경로 =', VAL_DIR)
print('test 경로 =', TEST_DIR)

if not TRAIN_DIR.exists() or not VAL_DIR.exists() or not TEST_DIR.exists():
    raise FileNotFoundError('split_dataset 폴더를 찾을 수 없습니다.')

In [ ]:
# EfficientNet-B0 입력 크기와 데이터 로더 설정입니다.
IMAGE_SIZE = 224
BATCH_SIZE = 16
NUM_WORKERS = 0
USE_PRETRAINED = True

normalize_mean = [0.485, 0.456, 0.406]
normalize_std = [0.229, 0.224, 0.225]

# train에는 augmentation을 적용하고, validation/test에는 고정 전처리만 사용합니다.
train_transform = transforms.Compose([
    transforms.RandomResizedCrop(IMAGE_SIZE, scale=(0.85, 1.0), ratio=(0.9, 1.1)),
    transforms.RandomHorizontalFlip(p=0.5),
    transforms.RandomRotation(degrees=10),
    transforms.ColorJitter(brightness=0.2, contrast=0.2),
    transforms.ToTensor(),
    transforms.Normalize(mean=normalize_mean, std=normalize_std),
])

eval_transform = transforms.Compose([
    transforms.Resize((IMAGE_SIZE, IMAGE_SIZE)),
    transforms.ToTensor(),
    transforms.Normalize(mean=normalize_mean, std=normalize_std),
])

train_dataset = datasets.ImageFolder(TRAIN_DIR, transform=train_transform)
val_dataset = datasets.ImageFolder(VAL_DIR, transform=eval_transform)
test_dataset = datasets.ImageFolder(TEST_DIR, transform=eval_transform)

# 클래스 불균형을 줄이기 위해 WeightedRandomSampler를 사용합니다.
train_class_counts = torch.bincount(torch.tensor(train_dataset.targets), minlength=len(train_dataset.classes)).float()
class_weights = train_class_counts.sum() / (len(train_class_counts) * train_class_counts)
sample_weights = [class_weights[target].item() for target in train_dataset.targets]
train_sampler = WeightedRandomSampler(weights=sample_weights, num_samples=len(sample_weights), replacement=True)

train_loader = DataLoader(train_dataset, batch_size=BATCH_SIZE, sampler=train_sampler, num_workers=NUM_WORKERS)
val_loader = DataLoader(val_dataset, batch_size=BATCH_SIZE, shuffle=False, num_workers=NUM_WORKERS)
test_loader = DataLoader(test_dataset, batch_size=BATCH_SIZE, shuffle=False, num_workers=NUM_WORKERS)

print('클래스 목록 =', train_dataset.classes)
print('train 클래스 개수 =', train_class_counts.tolist())

In [ ]:
# pretrained weight 캐시를 프로젝트 내부에 두어 경로 문제를 줄입니다.
LOCAL_TORCH_HOME = Y_MODEL_DIR / 'torch_cache'
LOCAL_TORCH_HOME.mkdir(parents=True, exist_ok=True)
os.environ['TORCH_HOME'] = str(LOCAL_TORCH_HOME)

try:
    weights = EfficientNet_B0_Weights.DEFAULT if USE_PRETRAINED else None
    model = efficientnet_b0(weights=weights)
except Exception as exc:
    print('사전학습 가중치를 불러오지 못해 weights=None으로 진행합니다.')
    print('원인 =', exc)
    model = efficientnet_b0(weights=None)

num_features = model.classifier[1].in_features
model.classifier[1] = nn.Linear(num_features, len(train_dataset.classes))
model = model.to(device)

print('마지막 분류기 =', model.classifier[1])

In [ ]:
# 손실 함수와 옵티마이저 설정입니다.
criterion = nn.CrossEntropyLoss(weight=class_weights.to(device), label_smoothing=0.1)
optimizer = torch.optim.AdamW(model.parameters(), lr=1e-4, weight_decay=1e-4)
EPOCHS = 10

In [ ]:
# optimizer가 있으면 학습, 없으면 평가 모드로 한 epoch를 실행합니다.
def run_epoch(model, loader, criterion, optimizer=None):
    is_train = optimizer is not None
    if is_train:
        model.train()
    else:
        model.eval()

    running_loss = 0.0
    correct = 0
    total = 0

    for images, labels in loader:
        images = images.to(device)
        labels = labels.to(device)

        if is_train:
            optimizer.zero_grad()

        with torch.set_grad_enabled(is_train):
            logits = model(images)
            loss = criterion(logits, labels)

            if is_train:
                loss.backward()
                optimizer.step()

        running_loss += loss.item() * images.size(0)
        preds = logits.argmax(dim=1)
        correct += (preds == labels).sum().item()
        total += labels.size(0)

    return running_loss / total, correct / total


In [ ]:
# validation accuracy가 가장 좋았던 가중치를 따로 보관합니다.
best_val_acc = 0.0
best_state_dict = copy.deepcopy(model.state_dict())
history = []

for epoch in range(1, EPOCHS + 1):
    train_loss, train_acc = run_epoch(model, train_loader, criterion, optimizer)
    val_loss, val_acc = run_epoch(model, val_loader, criterion)

    history.append({
        'epoch': epoch,
        'train_loss': train_loss,
        'train_acc': train_acc,
        'val_loss': val_loss,
        'val_acc': val_acc,
    })

    print(f'Epoch {epoch:02d} | train_loss={train_loss:.4f} train_acc={train_acc:.4f} | val_loss={val_loss:.4f} val_acc={val_acc:.4f}')

    if val_acc > best_val_acc:
        best_val_acc = val_acc
        best_state_dict = copy.deepcopy(model.state_dict())

model.load_state_dict(best_state_dict)
print('최고 validation accuracy =', best_val_acc)

In [ ]:
test_loss, test_acc = run_epoch(model, test_loader, criterion)
print('test loss =', round(test_loss, 4))
print('test accuracy =', round(test_acc, 4))

In [ ]:
# 학습이 끝난 뒤 체크포인트를 pth/pt 두 형식으로 저장합니다.
MODEL_PTH_PATH = Y_MODEL_DIR / 'efficientnet_b0_drowsiness.pth'
MODEL_PT_PATH = Y_MODEL_DIR / 'efficientnet_b0_drowsiness.pt'

checkpoint = {
    'model_name': 'efficientnet_b0',
    'model_state_dict': model.state_dict(),
    'class_names': train_dataset.classes,
    'image_size': IMAGE_SIZE,
    'normalize_mean': normalize_mean,
    'normalize_std': normalize_std,
    'best_val_acc': best_val_acc,
    'test_acc': test_acc,
    'history': history,
}

torch.save(checkpoint, MODEL_PTH_PATH)
torch.save(checkpoint, MODEL_PT_PATH)

print('저장 완료 =', MODEL_PTH_PATH)
print('저장 완료 =', MODEL_PT_PATH)

In [ ]:
# 실시간 추론 시에는 EfficientNet 체크포인트를 우선 사용하도록 환경 변수를 지정합니다.
REALTIME_PT_PATH = Y_MODEL_DIR / 'efficientnet_b0_drowsiness.pt'
REALTIME_PTH_PATH = Y_MODEL_DIR / 'efficientnet_b0_drowsiness.pth'

print('REALTIME_PT_PATH 존재 여부 =', REALTIME_PT_PATH.exists())
print('REALTIME_PTH_PATH 존재 여부 =', REALTIME_PTH_PATH.exists())

os.environ['DROWSINESS_CHECKPOINT_PATH'] = str(REALTIME_PT_PATH if REALTIME_PT_PATH.exists() else REALTIME_PTH_PATH)
if str(PROJECT_ROOT_DIR) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT_DIR))

from main import run_realtime

In [ ]:
# 학습한 EfficientNet-B0 체크포인트를 사용해 OpenCV 실시간 추론을 실행합니다.
# OpenCV 창에서 q 키를 누르면 종료됩니다.
run_realtime(
    camera_index=0,
    drowsy_threshold_frames=8,
    min_confidence=0.6,
)